# Root Cause Analysis (RCA) - Getting Started Notebook

This notebook will help you:
1. Load and explore the scenario data
2. Understand the signal structure
3. Visualize telemetry patterns
4. Get started with your RCA implementation

**Before running:** Make sure you're in the project directory and have installed required packages.

## Setup

First, let's install required packages and set up the environment.

In [ ]:
# Install required packages (uncomment if needed)
# !pip install pandas numpy matplotlib scikit-learn networkx ruptures statsmodels

In [ ]:
import sys
import os

# Add starter_code to path
sys.path.insert(0, '../starter_code')

# Standard imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json

# Project imports
from data_loader import (
    list_scenarios, load_scenario, load_signals_metadata,
    load_domain_knowledge, get_numeric_signals
)
from visualization import (
    plot_signal, plot_multiple_signals, plot_correlation_matrix,
    create_scenario_summary_plot
)

# Display settings
pd.set_option('display.max_columns', 50)
plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

print("Setup complete!")

## 1. Explore Available Scenarios

Let's see what scenarios are available for analysis.

In [ ]:
# List all scenarios
scenarios = list_scenarios()
print(f"Available scenarios ({len(scenarios)}):")
for s in scenarios:
    print(f"  - {s}")

In [ ]:
# Load signal metadata
signals_metadata = load_signals_metadata()

print(f"Total signals: {len(signals_metadata['signals'])}")
print(f"Sampling rate: {signals_metadata['sampling_rate_hz']} Hz")
print(f"Subsystems: {signals_metadata['subsystems']}")

## 2. Load and Explore a Scenario

Let's load the first easy scenario and explore the data.

In [ ]:
# Load scenario 1 (Cooling Pump Failure - Easy)
scenario_name = 'scenario_01_easy'
df, incident, ground_truth = load_scenario(scenario_name)

print(f"Scenario: {incident['scenario_name']}")
print(f"Difficulty: {incident['difficulty']}")
print(f"Alert Type: {incident['alert_type']}")
print(f"Description: {incident['description']}")

In [ ]:
# Examine the telemetry data
print(f"\nData shape: {df.shape}")
print(f"Time range: {df['timestamp'].min()} to {df['timestamp'].max()}")
print(f"Duration: {(df['timestamp'].max() - df['timestamp'].min()).total_seconds() / 60:.1f} minutes")
print(f"\nSignals ({len(get_numeric_signals(df))}):")
for i, sig in enumerate(get_numeric_signals(df)[:10]):
    print(f"  {sig}")
print("  ...")

In [ ]:
# Look at the ground truth (for learning - don't peek during evaluation!)
print("Ground Truth (for reference):")
print(f"  Root Cause: {ground_truth['root_cause']}")
print(f"  Root Cause Time: {ground_truth['root_cause_timestamp']}")
print(f"\nCausal Chain:")
for item in ground_truth['causal_chain']:
    print(f"  {item['order']}. {item['signal']} ({item['event']})")

## 3. Visualize the Data

Let's visualize the telemetry to understand the failure pattern.

In [ ]:
# Plot the root cause signal
root_cause = ground_truth['root_cause']
fig, ax = plt.subplots(figsize=(14, 4))
plot_signal(df, root_cause, ax=ax, title=f"Root Cause Signal: {root_cause}")
plt.tight_layout()
plt.show()

In [ ]:
# Plot signals in the causal chain
chain_signals = [item['signal'] for item in ground_truth['causal_chain'][:5]]
fig = plot_multiple_signals(df, chain_signals, title="Causal Chain Signals")
plt.show()

In [ ]:
# Plot correlation matrix for cooling system signals
cooling_signals = ['PUMP_COOLANT_FLOW', 'PUMP_CURRENT', 'COOLANT_FLOW_RATE', 
                   'TEMP_COOLANT_IN', 'TEMP_COOLANT_OUT', 'FAN_SPEED_RADIATOR']
fig = plot_correlation_matrix(df, cooling_signals, figsize=(8, 6))
plt.show()

## 4. Basic Anomaly Detection

Let's try some basic anomaly detection to understand the data.

In [ ]:
def detect_anomalies_zscore(signal, threshold=3.0):
    """Simple z-score based anomaly detection."""
    z_scores = (signal - signal.mean()) / signal.std()
    return np.abs(z_scores) > threshold

# Detect anomalies in root cause signal
anomalies = detect_anomalies_zscore(df[root_cause])
anomaly_indices = np.where(anomalies)[0]

print(f"Anomalies detected in {root_cause}: {len(anomaly_indices)} points")
if len(anomaly_indices) > 0:
    first_anomaly_idx = anomaly_indices[0]
    first_anomaly_time = df['timestamp'].iloc[first_anomaly_idx]
    print(f"First anomaly at index {first_anomaly_idx}, time: {first_anomaly_time}")

In [ ]:
# Find first anomaly for all signals
first_anomaly_times = {}
numeric_signals = get_numeric_signals(df)

for sig in numeric_signals:
    anomalies = detect_anomalies_zscore(df[sig])
    if anomalies.any():
        first_idx = np.where(anomalies)[0][0]
        first_anomaly_times[sig] = first_idx

# Sort by first anomaly time
sorted_signals = sorted(first_anomaly_times.items(), key=lambda x: x[1])

print("Signals sorted by first anomaly time:")
for sig, idx in sorted_signals[:10]:
    time = df['timestamp'].iloc[idx]
    print(f"  {sig}: index {idx} ({time.strftime('%H:%M:%S')})")

## 5. Domain Knowledge

Let's explore the domain knowledge that can help with RCA.

In [ ]:
# Load domain knowledge
domain = load_domain_knowledge()

print("Subsystems:")
for name, info in domain['subsystems'].items():
    print(f"\n{name.upper()}:")
    print(f"  Description: {info['description']}")
    print(f"  Failure modes: {info['failure_modes'][:2]}...")

In [ ]:
# Look at known causal relationships
print("Known Causal Relationships:")
for rel in domain['causal_relationships']['relationships'][:5]:
    print(f"  {rel['cause']} → {rel['effect']}")
    print(f"    Type: {rel['relationship']}, Lag: {rel['typical_lag_seconds']}s")

## 6. Try a Baseline

Let's run a baseline algorithm to see how it performs.

In [ ]:
# Add baselines to path and import
sys.path.insert(0, '../baselines')
from baseline_2_zscore import identify_root_cause as zscore_rca

# Run baseline
result = zscore_rca(df)

print("Z-Score Temporal Baseline Results:")
print(f"  Predicted Root Cause: {result['root_cause']}")
print(f"  Confidence: {result['confidence']:.3f}")
print(f"  Actual Root Cause: {ground_truth['root_cause']}")
print(f"  Correct: {result['root_cause'] == ground_truth['root_cause']}")

## 7. Your Implementation

Now it's your turn! Use the cells below to start developing your RCA solution.

Remember the 5 steps:
1. **Incident Window Isolation** - Find the anomaly window
2. **Signal Clustering** - Group related signals
3. **Temporal Analysis** - Determine who changed first
4. **Causal Graph** - Build cause-effect relationships
5. **Ranking & Explanation** - Identify and explain root cause

In [ ]:
# Your Step 1: Incident Window Isolation
# TODO: Implement change-point detection or anomaly-based window detection

# Example: use ruptures library
# import ruptures as rpt
# ...

pass

In [ ]:
# Your Step 2: Signal Clustering
# TODO: Extract features and cluster signals

# Example: use sklearn clustering
# from sklearn.cluster import KMeans
# ...

pass

In [ ]:
# Your Step 3: Temporal Analysis
# TODO: Compute lagged correlations and Granger causality

# Example: use statsmodels
# from statsmodels.tsa.stattools import grangercausalitytests
# ...

pass

In [ ]:
# Your Step 4: Causal Graph Construction
# TODO: Build directed graph from temporal relationships

# Example: use networkx
# import networkx as nx
# ...

pass

In [ ]:
# Your Step 5: Root Cause Ranking
# TODO: Rank candidates and generate explanations

pass

## 8. Evaluate Your Solution

Run your solution on all scenarios and evaluate.

In [ ]:
# When you have a complete solution, use the evaluation script:
# 1. Save your results to a JSON file
# 2. Run: python ../evaluation/evaluate.py your_results.json

# Or use the metrics directly:
sys.path.insert(0, '../evaluation')
from metrics import compute_all_metrics, print_metrics_report

# Example (replace with your actual result):
# metrics = compute_all_metrics(your_result, ground_truth)
# print_metrics_report(metrics)

## Next Steps

1. Implement each step of the pipeline
2. Test on easy scenarios first
3. Iterate and improve your algorithms
4. Run on all scenarios and evaluate
5. Document your approach and decisions

Good luck!